In [ ]:
from sympy import *

# ------------------------------------------------------------
# General circuit solver for node equations in Laplace domain
# ------------------------------------------------------------

def solve_node_circuit(equations, nodes, input_signal=None, output_node=None, s_symbol=None):
    """
    Solves node equations in the Laplace domain.

    Parameters
    ----------
    equations : list
        List of SymPy equations, e.g. [eqA, eqB, eqC].

    nodes : list
        Unknown node voltages, e.g. [VA, VB] or [VA, VB, VC].

    input_signal : SymPy symbol, optional
        Known input signal, e.g. V1.

    output_node : SymPy symbol, optional
        Desired output node, e.g. VB.

    s_symbol : SymPy symbol, optional
        Laplace variable. Defaults to global symbol s.

    Returns
    -------
    sol : dict
        Solutions for all node voltages.

    H : SymPy expression or None
        Transfer function output_node/input_signal if both are given.
    """

    if s_symbol is None:
        s_symbol = symbols('s')

    # Solve equations
    sol_list = solve(equations, nodes, dict=True)

    if not sol_list:
        print("No solution found.")
        return None, None

    sol = sol_list[0]

    print("Isolated node voltages:")
    print("-----------------------")

    for node in nodes:
        if node in sol:
            expr = together(simplify(sol[node]))
            num, den = fraction(expr)
            expr_clean = collect(expand(num), s_symbol) / collect(expand(den), s_symbol)

            print(f"{node} =")
            display(expr_clean)
            print()

    H = None

    # Transfer function if requested
    if input_signal is not None and output_node is not None:
        if output_node not in sol:
            print(f"{output_node} was not solved.")
            return sol, None

        H = simplify(sol[output_node] / input_signal)
        H = together(H)

        num, den = fraction(H)

        num = collect(expand(num), s_symbol)
        den = collect(expand(den), s_symbol)

        H = num / den

        print("Transfer function:")
        print("------------------")
        print(f"H(s) = {output_node}/{input_signal} =")
        display(H)
        print()

        print("Numerator:")
        display(num)
        print()

        print("Denominator:")
        display(den)
        print()

        # Coefficients
        num_poly = Poly(num, s_symbol)
        den_poly = Poly(den, s_symbol)

        b_coeffs = num_poly.all_coeffs()
        a_coeffs = den_poly.all_coeffs()

        print("Numerator coefficients:")
        display(b_coeffs)
        print()

        print("Denominator coefficients:")
        display(a_coeffs)
        print()

        # Laplace equation
        X, Y = symbols('X Y')
        laplace_eq = Eq(den * Y, num * X)

        print("Laplace-domain equation:")
        display(laplace_eq)
        print()

        print("Differential equation rule:")
        print("If")
        display(laplace_eq)
        print("then replace:")
        print("s**2*Y -> y''(t),  s*Y -> y'(t),  Y -> y(t)")
        print("s**2*X -> x''(t),  s*X -> x'(t),  X -> x(t)")

    return sol, H

In [ ]:
s = symbols('s')

R1, R2, C1, C2, L1, L2 = symbols('R1 R2 C1 C2 L1 L2', positive=True, real=True)

VA, VB, VC, V1 = symbols('VA VB VC V1')

eqA = Eq((VA-V1)/R1 + s*C1*VA + (VA-VB)/R2,0)
eqB = Eq((VB-VA)/R2 + VB/(s*L1), 0)
sol, H = solve_node_circuit(
    equations=[eqA, eqB],
    nodes=[VA,VB],
    input_signal=V1,
    output_node=VB,
    s_symbol=s
    )

Isolated node voltages:
-----------------------
VA =


(L1*V1*s + R2*V1)/(C1*L1*R1*s**2 + R1 + R2 + s*(C1*R1*R2 + L1))


VB =


L1*V1*s/(C1*L1*R1*s**2 + R1 + R2 + s*(C1*R1*R2 + L1))


Transfer function:
------------------
H(s) = VB/V1 =


...


Laplace-domain equation:


Eq(Y*(C1*L1*R1*s**2 + R1 + R2 + s*(C1*R1*R2 + L1)), L1*X*s)


Differential equation rule:
If


Eq(Y*(C1*L1*R1*s**2 + R1 + R2 + s*(C1*R1*R2 + L1)), L1*X*s)

then replace:
s**2*Y -> y''(t),  s*Y -> y'(t),  Y -> y(t)
s**2*X -> x''(t),  s*X -> x'(t),  X -> x(t)


In [5]:
s = symbols('s')

R1, R2, C1, C2, L1, L2 = symbols('R1 R2 C1 C2 L1 L2', positive=True, real=True)

VA, VB, VC, V1 = symbols('VA VB VC V1')

eqA = Eq((VA - V1) * s * C1 + VA / R1 + (VA - VB) / R2,0)
eqB = Eq(((VB - VA)/R2 + (VB)* s * C2), 0)
sol, H = solve_node_circuit(
    equations=[eqA, eqB],
    nodes=[VA,VB],
    input_signal=V1,
    output_node=VB,
    s_symbol=s
    )



Isolated node voltages:
-----------------------
VA =


(C1*C2*R1*R2*V1*s**2 + C1*R1*V1*s)/(C1*C2*R1*R2*s**2 + s*(C1*R1 + C2*R1 + C2*R2) + 1)


VB =


C1*R1*V1*s/(C1*C2*R1*R2*s**2 + s*(C1*R1 + C2*R1 + C2*R2) + 1)


Transfer function:
------------------
H(s) = VB/V1 =


...


Laplace-domain equation:


Eq(Y*(C1*C2*R1*R2*s**2 + s*(C1*R1 + C2*R1 + C2*R2) + 1), C1*R1*X*s)


Differential equation rule:
If


Eq(Y*(C1*C2*R1*R2*s**2 + s*(C1*R1 + C2*R1 + C2*R2) + 1), C1*R1*X*s)

then replace:
s**2*Y -> y''(t),  s*Y -> y'(t),  Y -> y(t)
s**2*X -> x''(t),  s*X -> x'(t),  X -> x(t)
